In [ ]:
import json

with open("chunks_v2.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

In [ ]:
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print(embeddings.shape)

In [ ]:
import numpy as np

np.save("embeddings.npy", embeddings)
print("Saved embeddings.npy")

In [ ]:
def search(questions, top_k=5):
  question_embedding = model.encode(questions)
  scores = util.cos_sim(question_embedding, embeddings)[0]
  top_results = scores.argsort(descending=True)[:top_k]

  for rank, idx in enumerate(top_results):
    idx = int(idx)
    print(f"Rank: {rank+1}")
    print(f"ID: {chunks[idx]['id']}")
    print(f"Score: {scores[idx]:.4f}")
    print(f"Section: {chunks[idx]['section']}")
    print(f"Text: {chunks[idx]['text'][:130]}...")
    print()

In [ ]:
search("What is the maximum duration to complete the Bachelor's degree?")

In [ ]:
eval_set = [
    {"question": "How many credit hours are required for the B.Sc. in Nursing?", "correct_id": 1},
    {"question": "What is the maximum duration to complete the Bachelor's degree?", "correct_id": 1},
    {"question": "What is the minimum and maximum credit hours I can register in a regular semester?", "correct_id": 2},
    {"question": "How many credit hours can I register in summer?", "correct_id": 5},
    {"question": "What percentage of classes can I miss before being barred from the exam?", "correct_id": 9},
    {"question": "What happens if my absence exceeds 20% due to an accepted excuse?", "correct_id": 11},
    {"question": "How soon after enrollment can a newly enrolled student postpone their studies?", "correct_id": 30},
    {"question": "How is the final grade of a course calculated?", "correct_id": 40},
    {"question": "What does nursing ethics refer to?", "correct_id": 90},
    {"question": "How should a request to change specialization be submitted?", "correct_id": 21},
    {"question": "Who decides on requests to change specialization, and what is it subject to?", "correct_id": 20},
    {"question": "Does the semester following a dismissal count toward the maximum graduation period?", "correct_id": 35},
    {"question": "Can Arab students change to a specialization with a lower admission average?", "correct_id": 25},
]


In [ ]:
def get_top_k_ids(question, top_k=5):
    question_embedding = model.encode(question)
    scores = util.cos_sim(question_embedding, embeddings)[0]
    top_results = scores.argsort(descending=True)[:top_k]
    return [int(chunks[i]['id']) for i in top_results]

In [ ]:
hits_at_1, hits_at_3, hits_at_5 = 0, 0, 0

for item in eval_set:
    retrieved_ids = get_top_k_ids(item["question"], top_k=5)
    correct = item["correct_id"]
    in_top1 = correct in retrieved_ids[:1]
    in_top3 = correct in retrieved_ids[:3]
    in_top5 = correct in retrieved_ids[:5]
    hits_at_1 += in_top1
    hits_at_3 += in_top3
    hits_at_5 += in_top5
    print(f"Q: {item['question']}")
    print(f"Expected ID: {correct} | Retrieved IDs: {retrieved_ids}")
    print(f"Top1: {'✓' if in_top1 else '✗'}  Top3: {'✓' if in_top3 else '✗'}  Top5: {'✓' if in_top5 else '✗'}\n")

n = len(eval_set)
print(f"Recall@1: {hits_at_1/n:.2f}  ({hits_at_1}/{n})")
print(f"Recall@3: {hits_at_3/n:.2f}  ({hits_at_3}/{n})")
print(f"Recall@5: {hits_at_5/n:.2f}  ({hits_at_5}/{n})")

loading the API code safely :

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

print("Key loaded:", api_key is not None)

configuring the Gemini client :
didn't actually call gemini just prepared it 

In [ ]:



import google.generativeai as genai

genai.configure(api_key=api_key)
llm = genai.GenerativeModel("gemini-3.5-flash-lite")

print("Gemini ready")

The function tying retrieval and generation together 
— the actual RAG step

In [ ]:
def answer_question(question, top_k=5):
    retrieved_ids = get_top_k_ids(question, top_k=top_k)
    retrieved_chunks = [c for c in chunks if c["id"] in retrieved_ids]

    context = "\n\n".join(
        f"[Chunk {c['id']}] {c['text']}" for c in retrieved_chunks
    )

    prompt = f"""You are answering questions about the JUST Faculty of Nursing Student Handbook.
Answer the question using ONLY the information in the passages below.
If the passages don't contain the answer, say "I don't have enough information to answer that."
Cite the chunk ID(s) you used in your answer.

Passages:
{context}

Question: {question}

Answer:"""

    response = llm.generate_content(prompt)
    return response.text

First single test:

In [ ]:
answer = answer_question("What is the maximum duration to complete the Bachelor's degree?")
print(answer)

Testing generation across multiple questions, with rate-limit handling

In [ ]:
import time

for item in eval_set[:6]:
    print(f"Q: {item['question']}")
    try:
        answer = answer_question(item['question'])
        print(f"A: {answer}")
        has_citation = "[Chunk" in answer
        print(f"Citation present: {'✓' if has_citation else '✗'}")
    except Exception as e:
        print(f"Request failed (likely rate limit): {e}")
    print()
    time.sleep(10)

the interface lololoeeesh

In [ ]:


import gradio as gr
import time

def gradio_answer(question):
    if not question.strip():
        return "Please enter a question."
    for attempt in range(3):
        try:
            return answer_question(question)
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(15)
                continue
            return f"Something went wrong: {e}"

demo=gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(label="Ask about the JUST Nursing Student Handbook", placeholder="e.g. How many credit hours can I register in summer?"),
    outputs=gr.Textbox(label="Answer"),
    title="JUST Nursing Handbook Q&A",
    description="Ask a question about academic regulations, attendance, or clinical guidelines . Answers are grounded in the handbook and cite the source chunk."
)

demo.launch()
